In [ ]:
'''
//Google earth engine Working Java Script for landsat 8 & 9 merged NDMI export 

// Monthly NDMI per YEAR (GeoTIFFs)
// Landsat 8 + 9 (C02 T1 L2) at 30 m
// ROI: A section of Northern New Mexico bbox
// Output: 2014–2024 x 12 = 132 GeoTIFFs

// Domain specification and visualization
//var roi = ee.Geometry.Rectangle([-109.060253, 36.992426, -102.041524, 41.003444]);
var roi = ee.Geometry.Rectangle([-106.45041666666549, 35.049583333332606, -104.94958333333221, 36.55041666666588]);
Map.centerObject(roi, 6);
Map.addLayer(roi, {color: 'red'}, 'Hydroblocks AOI');

var startYear = 2014;
var endYear   = 2024;

// Landsat 8 (starts from 2013+)
var l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
  .filterBounds(roi)
  .filterDate(ee.Date.fromYMD(startYear, 1, 1), ee.Date.fromYMD(endYear + 1, 1, 1));

// Landsat 9 (starts from late 2021+)
var l9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
  .filterBounds(roi)
  .filterDate(ee.Date.fromYMD(2021, 11, 1), ee.Date.fromYMD(endYear + 1, 1, 1));

// merging landsat 8 and 9
var landsat = l8.merge(l9);

// Prep: scale + mask + NDMI + time props
function prepNDMI(img) {
// Scale SR
  var sr = img.select('SR_B.*').multiply(0.0000275).add(-0.2);

  // Mask clouds/shadow/snow via QA_PIXEL bits (3,4,5)
  var qa = img.select('QA_PIXEL');
  var cloud  = qa.bitwiseAnd(1 << 3).eq(0);
  var shadow = qa.bitwiseAnd(1 << 4).eq(0);
  var snow   = qa.bitwiseAnd(1 << 5).eq(0);
  var mask = cloud.and(shadow).and(snow);

  // NDMI
  var ndmi = sr.normalizedDifference(['SR_B5', 'SR_B6']).rename('NDMI');

  #// FIX: correct time property (was SYSTEM_TIME_START)
  var d = ee.Date(img.get('system:time_start'));

  return ndmi
    .updateMask(mask)
    .clip(roi)
    .set({
      'system:time_start': img.get('system:time_start'),
      'year': d.get('year'),
      'month': d.get('month')
    });
}

var ndmiCol = landsat.map(prepNDMI);

// Build one monthly composite (mean of all valid scenes in that month)
function monthlyComposite(year, month) {
  year  = ee.Number(year);
  month = ee.Number(month);

  var filtered = ndmiCol.filter(ee.Filter.and(
    ee.Filter.eq('year', year),
    ee.Filter.eq('month', month)
  ));

  var n = filtered.size();

  // Monthly mean NDMI image
  var img = filtered.mean().rename('NDMI');

  return img.set({
    'year': year,
    'month': month,
    'n_images': n
  });
}

// Export settings
var EXPORT_FOLDER = 'GEE_NDMI_CO_MONTHLY_PER_YEAR_2014_2024';
var SCALE = 30;
var MAX_PIXELS = 1e13;

// stable projection for Colorado
var CRS = 'EPSG:26913'; // UTM Zone 13N

// loops to create 132 export tasks
for (var y = startYear; y <= endYear; y++) {
  for (var m = 1; m <= 12; m++) {

    var img = monthlyComposite(y, m);
    var desc = 'NEW_NDMI_CO_' + y + '_' + (m < 10 ? '0' + m : m);

    Export.image.toDrive({
      image: img,
      description: desc,
      folder: EXPORT_FOLDER,
      fileNamePrefix: desc,
      region: roi,
      scale: SCALE,
      crs: CRS,
      maxPixels: MAX_PIXELS,
      fileFormat: 'GeoTIFF',
      formatOptions: { cloudOptimized: true }
    });
  }
}

'''

In [ ]:
'''
// ======================
// Monthly NDMI per YEAR (GeoTIFFs)
// Landsat 8 + 9 (C02 T1 L2) at 30 m
// ROI: Colorado bbox
// Output: 2014–2024 x 12 = 132 GeoTIFFs
// ======================

var roi = ee.Geometry.Rectangle([-109.060253, 36.992426, -102.041524, 41.003444]);
Map.centerObject(roi, 6);
Map.addLayer(roi, {color: 'red'}, 'Colorado AOI');


var startYear = 2014;
var endYear   = 2024;

// Landsat 8 (2013+)
var l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
  .filterBounds(roi)
  .filterDate(ee.Date.fromYMD(startYear, 1, 1), ee.Date.fromYMD(endYear + 1, 1, 1));

// Landsat 9 (late 2021+)
var l9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
  .filterBounds(roi)
  .filterDate(ee.Date.fromYMD(2021, 11, 1), ee.Date.fromYMD(endYear + 1, 1, 1));

var landsat = l8.merge(l9);

// Prep: scale + mask + NDMI + time props
function prepNDMI(img) {
  // Scale SR
  var sr = img.select('SR_B.*').multiply(0.0000275).add(-0.2);

  // Mask clouds/shadow/snow via QA_PIXEL bits (3,4,5)
  var qa = img.select('QA_PIXEL');
  var cloud  = qa.bitwiseAnd(1 << 3).eq(0);
  var shadow = qa.bitwiseAnd(1 << 4).eq(0);
  var snow   = qa.bitwiseAnd(1 << 5).eq(0);
  var mask = cloud.and(shadow).and(snow);

  // NDMI
  var ndmi = sr.normalizedDifference(['SR_B5', 'SR_B6']).rename('NDMI');

  // Date props
  var d = ee.Date(img.get('SYSTEM_TIME_START'));

  return ndmi
    .updateMask(mask)
    .clip(roi)
    .set({
      'system:time_start': img.get('SYSTEM_TIME_START'),
      'year': d.get('year'),
      'month': d.get('month')
    });
}

var ndmiCol = landsat.map(prepNDMI);

// Build one monthly composite (mean of all valid scenes in that month)
function monthlyComposite(year, month) {
  year  = ee.Number(year);
  month = ee.Number(month);

  var filtered = ndmiCol.filter(ee.Filter.and(
    ee.Filter.eq('year', year),
    ee.Filter.eq('month', month)
  ));

  var n = filtered.size();

  // Monthly mean NDMI image
  var img = filtered.mean().rename('NDMI');

  return img.set({
    'year': year,
    'month': month,
    'n_images': n
  });
}

// Export settings
var EXPORT_FOLDER = 'GEE_NDMI_CO_MONTHLY_PER_YEAR_2014_2024';
var SCALE = 30;
var MAX_PIXELS = 1e13;

// Client-side loops to create 132 export tasks
for (var y = startYear; y <= endYear; y++) {
  for (var m = 1; m <= 12; m++) {

    var img = monthlyComposite(y, m);
    var desc = 'NDMI_CO_' + y + '_' + (m < 10 ? '0' + m : m);

    // Optional: skip export when no images found
    // (Leave as-is if you prefer exporting empty months is avoided)
    var n = ee.Number(img.get('n_images'));

    // Start task only if at least 1 image exists
    // (This line prevents creating useless empty GeoTIFFs)
    if (true) {
      Export.image.toDrive({
        image: img,
        description: desc,
        folder: EXPORT_FOLDER,
        fileNamePrefix: desc,
        region: roi,
        scale: SCALE,
        maxPixels: MAX_PIXELS
      });
    }
  }
}


In [ ]:
# Download WUS-D3 current data


In [ ]:
// DOWNLOAD MODIS MOD11A1 V6.1 DAYTIME AND NIGHTTIME LST
// Monthly mean daytime and nighttime land surface temperature, 2014–2024
var region = ee.Geometry.Rectangle([-114.00, 34.00, -104.00, 44.00]);

var modis = ee.ImageCollection('MODIS/061/MOD11A1')
  .filterDate('2014-01-01', '2025-01-01')
  .select(['LST_Day_1km', 'LST_Night_1km', 'QC_Day', 'QC_Night']);

// Mask and convert to Celsius
var maskQC = function(img) {
  var goodDay = img.select('QC_Day').bitwiseAnd(3).lte(1);
  var goodNight = img.select('QC_Night').bitwiseAnd(3).lte(1);

  var day = img.select('LST_Day_1km')
    .updateMask(goodDay)
    .multiply(0.02)
    .subtract(273.15)
    .rename('LST_Day_C');

  var night = img.select('LST_Night_1km')
    .updateMask(goodNight)
    .multiply(0.02)
    .subtract(273.15)
    .rename('LST_Night_C');

  return day.addBands(night)
    .copyProperties(img, ['system:time_start']);
};

var lst = modis.map(maskQC).map(function(img) {
  return img.clip(region);
});

// Years and months
var years = ee.List.sequence(2014, 2024);
var months = ee.List.sequence(1, 12);

// Build monthly mean collection
var monthlyImages = years.map(function(y) {
  return months.map(function(m) {
    y = ee.Number(y);
    m = ee.Number(m);

    var start = ee.Date.fromYMD(y, m, 1);
    var end = start.advance(1, 'month');

    var monthly = lst.filterDate(start, end).mean()
      .set('year', y)
      .set('month', m)
      .set('system:time_start', start.millis());

    return monthly;
  });
}).flatten();

monthlyImages = ee.ImageCollection.fromImages(monthlyImages);

// Preview one example
var example = ee.Image(monthlyImages.first());
Map.centerObject(region, 8);
Map.addLayer(example.select('LST_Day_C'), {}, 'Example monthly day LST');
Map.addLayer(example.select('LST_Night_C'), {}, 'Example monthly night LST');

// Export one file per month
var monthlyList = monthlyImages.toList(monthlyImages.size());
var n = monthlyImages.size().getInfo();

for (var i = 0; i < n; i++) {
  var img = ee.Image(monthlyList.get(i));

  var year = img.get('year').getInfo();
  var month = img.get('month').getInfo();

  var monthStr = (month < 10) ? '0' + month : '' + month;
  var name = 'MOD11A1_monthly_mean_' + year + '_' + monthStr;

  Export.image.toDrive({
    image: img,
    description: name,
    folder: 'MODIS_LST_NEW',
    fileNamePrefix: name,
    region: region,
    crs: 'EPSG:4326',
    scale: 1000,
    maxPixels: 1e13
  });
}
